# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, baselines, MOO, figures, download.

**Run cells in order.** After Cell 2, the repo is checked out. After Cell 5, weights are ready.

In [22]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

Cloning into '2601_chip_paper'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 2), reused 29 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 585.81 KiB | 32.54 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/2601_chip_paper/2601_chip_paper


In [23]:
# Cell 2: Pull latest code and install all dependencies
!git pull
!uv sync  # Installs ALL deps from pyproject.toml including xgboost

Already up to date.
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 79 packages in 0.97ms
Prepared 13 packages in 29ms                                             
Installed 74 packages in 448ms                              
 + about-time==4.2.1
 + alive-progress==3.3.0
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + autograd==1.8.0
 + cffi==2.0.0
 + click==8.3.3
 + cma==4.4.4
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + deprecated==1.3.1
 + executing==2.2.1
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.2.0
 + graphemeu==0.7.2
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.5.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mdurl==0.1.2
 + moocore==0.3.1
 + mpmath==1.3.0
 + nest-async

In [24]:
# Cell 3: Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [25]:
# Cell 4: Monotonicity ground truth audit (produces paper Section 4.1 data)
# Quick mode: --max-groups 5000 for a fast sample, remove flag for full audit
!uv run python cli.py analysis monotonicity --max-groups 5000

Usage: cli.py [OPTIONS] COMMAND [ARGS]...
Try 'cli.py --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'analysis'.                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯


In [ ]:
# Cell 5: Train HINN (primary model, seed=42)
# For multi-seed reporting: re-run with --seed 0, 7, 123, 999
!uv run python cli.py train train --epochs 300 --batch-size 1024 --seed 42

In [ ]:
# Cell 6: Train baselines (XGBoost + Vanilla MLP) — same data split
!uv run python cli.py train baselines --epochs 300 --seed 42

In [ ]:
# Cell 7: Run MOO — extract Pareto front from trained HINN surrogate
!uv run python cli.py moo run

In [ ]:
# Cell 8: Generate all publication figures
!uv run python cli.py plot training-dynamics
!uv run python cli.py plot pareto
!uv run python cli.py plot monotonicity
!uv run python cli.py plot comparison

In [ ]:
# Cell 9: Zip all results (weights, scalers, CSVs, SVGs) and download
import shutil
from google.colab import files

# Include results/ subdirs: models/, figs/, data/
shutil.make_archive("hinn_results", "zip", "results")
files.download("hinn_results.zip")